# Cross-encoder reranking v3 — TPU edition

Same experiment as `reranking_v3.ipynb`, rewritten so the heavy work actually runs on a
**Colab TPU** via PyTorch/XLA.

### Two things about v3 that change on TPU

**1. v3's reason for existing was a CUDA problem that cannot occur here.** v3 was split off from v2
because `gte-multilingual-reranker-base` corrupted the CUDA context and took `mmarco-mMiniLM` down
with it, so v3 skips `bge-reranker-v2-m3` and resumes it from `rerank_results_partial.pkl` instead.
On an XLA/TPU backend there is no CUDA context to poison, and the "restart the runtime first"
instruction in the original is moot.

**2. A fresh runtime has no checkpoint.** `rerank_results_partial.pkl` lived in the session that
was restarted; it is not in the repo. Left alone, this notebook would produce a results table with
**no `bge-reranker-v2-m3` row at all** — dropping the very model that produced the headline
(Darija R@1 0.620 → 0.800). So:

- If a valid checkpoint is found (local or in Drive), completed models are reloaded and not rerun.
- If it is not found, `bge-reranker-v2-m3` is **added back to the run automatically**, so the
  table and the headline come out complete. The notebook prints which path it took.

**Checkpoint safety.** The original notebook's `results` dict holds per-item hit vectors scored
against a specific candidate set. v2's markdown reports a Darija baseline of 0.575 and v3's
reports 0.620 — retrieval differed between those runs, so a checkpoint from one is not comparable
to the other. On resume this notebook compares the checkpoint's `no_rerank` vectors against the
freshly computed ones and **discards mismatched reranker rows rather than silently mixing them.**

### What makes the TPU real

1. **Explicit XLA device**, with the backend actually obtained printed plainly.
2. **Fixed input shapes** — every batch padded to exactly `(batch_size, max_length)`, so XLA
   compiles one graph per model instead of recompiling per shape.
3. **Batched across queries** — all 4000 (query, candidate) pairs per field in one uniform stream.

Falls back to CUDA then CPU automatically if XLA is unavailable, and says so.

### Data

Fetched directly from the public GitHub repo — **nothing to upload**. `corpus_v2.json` is
committed in the repo as `data/corpus.json` (3054 chunks: 2974 `wikipedia_ar` + 80
`pilot_synthetic`).

`Runtime → Run all`.

### 1. Install PyTorch/XLA (matched to the runtime's existing torch, so no restart is needed)

In [1]:
import importlib.util, subprocess, sys

# Install torch_xla pinned to the ALREADY-INSTALLED torch version. Installing an
# unpinned torch_xla would drag in a different torch and force a runtime restart,
# which would wipe session state mid-notebook.
if importlib.util.find_spec("torch_xla") is None:
    import torch as _t
    _v = _t.__version__.split("+")[0]
    print(f"torch {_v} present, torch_xla missing -> installing torch_xla=={_v}")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={_v}",
         "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
        capture_output=True, text=True)
    print("pip exit", r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        print("\nInstall failed -- the notebook will fall back to CUDA/CPU and say so.")
else:
    print("torch_xla already available")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rank_bm25", "transformers", "sentencepiece"], check=False)
print("deps ready")

torch_xla already available
deps ready


### 2. Pick the device — and state plainly which one we actually got

In [2]:
import os, gc, json, re, pickle, time
import numpy as np
import pandas as pd
import torch

BACKEND = "cpu"
device = torch.device("cpu")
xm = None

try:
    import torch_xla
    import torch_xla.core.xla_model as _xm
    xm = _xm
    # torch_xla >= 2.5 prefers torch_xla.device(); older exposes xm.xla_device().
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    _ = (torch.ones(2, 2, device=device) * 2).sum().item()   # force a real TPU op
    BACKEND = "tpu"
except Exception as e:
    print(f"XLA unavailable ({type(e).__name__}: {str(e)[:160]})")
    if torch.cuda.is_available():
        device, BACKEND = torch.device("cuda"), "cuda"
    else:
        device, BACKEND = torch.device("cpu"), "cpu"


def sync():
    """Flush the XLA graph. No-op off TPU."""
    if BACKEND == "tpu":
        if hasattr(torch_xla, "sync"):
            torch_xla.sync()
        else:
            xm.mark_step()


print("=" * 78)
print(f"BACKEND ACTUALLY IN USE: {BACKEND.upper()}   (device={device})")
if BACKEND == "tpu":
    print(f"torch {torch.__version__} | torch_xla {torch_xla.__version__}")
    print("All encoding and reranking below runs on the TPU.")
else:
    print("NOT running on TPU. Results are still valid, just slower.")
print("=" * 78)

XLA unavailable (RuntimeError: TPU initialization failed: open(/dev/vfio/0): Device or resource busy: Device or resource busy; Couldn't open iommu group /dev/vfio/0)
BACKEND ACTUALLY IN USE: CPU   (device=cpu)
NOT running on TPU. Results are still valid, just slower.


### 3. Mount Drive up front

v3 is checkpoint-driven, so Drive is mounted **before** the run rather than after: it is where a
checkpoint from a previous session would be read from, and where this run's checkpoint is written
so the next session can resume from it.

In [3]:
DRIVE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_DIR = "/content/drive/MyDrive/MSA_rerank_v3_tpu"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Drive dir:", DRIVE_DIR)
except Exception as e:
    print(f"Drive not mounted ({type(e).__name__}: {str(e)[:120]}).")
    print("Checkpoints will be session-local only and lost on restart.")

# Search order for an existing checkpoint: working dir, this notebook's Drive
# folder, then the v2 TPU notebook's folder (a v2 run is a legitimate source).
CKPT_CANDIDATES = [p for p in [
    "rerank_results_partial.pkl",
    os.path.join(DRIVE_DIR, "rerank_results_partial.pkl") if DRIVE_DIR else None,
    "/content/drive/MyDrive/MSA_rerank_v2_tpu/rerank_results_partial.pkl" if DRIVE_DIR else None,
] if p]

found = [p for p in CKPT_CANDIDATES if os.path.exists(p)]
print("\nCheckpoint search:")
for p in CKPT_CANDIDATES:
    print(f"  {'FOUND  ' if os.path.exists(p) else 'missing'} {p}")
CKPT_PATH = found[0] if found else None
if CKPT_PATH is None:
    print("\nNo checkpoint. bge-reranker-v2-m3 will be added back to the run "
          "so the results table is complete.")

Mounted at /content/drive
Drive dir: /content/drive/MyDrive/MSA_rerank_v3_tpu

Checkpoint search:
  FOUND   rerank_results_partial.pkl
  missing /content/drive/MyDrive/MSA_rerank_v3_tpu/rerank_results_partial.pkl
  missing /content/drive/MyDrive/MSA_rerank_v2_tpu/rerank_results_partial.pkl


### 4. Config

In [ ]:
CONFIG = {
    "corpus_url": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/corpus.json",
    "wiki_qa_url": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "alpha": 0.8,

    # Candidates passed from retrieval into the reranker. Larger = higher
    # ceiling but slower, since the cross-encoder scores every candidate.
    "rerank_k": 20,

    # v3's own list: the two models that had not been measured yet.
    "rerankers": [
        "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",  # lightweight multilingual
        "BAAI/bge-reranker-base",                      # smaller BGE, size comparison vs v2-m3
        # jina-reranker-v2 and gte-multilingual-reranker-base stay dropped: jina's custom
        # code is incompatible with current transformers, and gte's CUDA fault is not worth
        # re-testing here even though XLA would not reproduce it.
    ],

    # Run only if no checkpoint supplies it. Without this, a fresh runtime yields a
    # table with no row for the model that produced the headline result.
    "completed_model": "BAAI/bge-reranker-v2-m3",
    "auto_include_completed": True,

    # Discard checkpointed reranker rows whose run used a different candidate set.
    "strict_resume": True,

    # Fixed for XLA: every batch is padded to exactly these dims so the graph
    # compiles once per model instead of once per distinct shape.
    "max_length": 512,
    "batch_size": 32,
    "encode_batch_size": 32,

    "k_values": (1, 3, 5, 10),
    "bootstrap_n": 1000,
    "seed": 42,
}
CONFIG

{'corpus_url': 'https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/corpus.json',
 'wiki_qa_url': 'https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data/qa_pairs_wiki.json',
 'base_encoder': 'intfloat/multilingual-e5-base',
 'alpha': 0.8,
 'rerank_k': 20,
 'rerankers': ['cross-encoder/mmarco-mMiniLMv2-L12-H384-v1',
  'BAAI/bge-reranker-base'],
 'completed_model': 'BAAI/bge-reranker-v2-m3',
 'auto_include_completed': True,
 'strict_resume': True,
 'max_length': 512,
 'batch_size': 32,
 'encode_batch_size': 32,
 'k_values': (1, 3, 5, 10),
 'bootstrap_n': 1000,
 'seed': 42}

### 5. Load data straight from GitHub

In [5]:
import urllib.request

def fetch_json(url):
    with urllib.request.urlopen(url) as r:
        return json.loads(r.read().decode("utf-8"))

corpus = fetch_json(CONFIG["corpus_url"])
wiki_qa = fetch_json(CONFIG["wiki_qa_url"])

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]

# No training happens here, so the full benchmark is used as the test set.
print(f"Corpus {len(corpus)} | evaluating on all {len(qa)} Wikipedia items")

Corpus 3054 | evaluating on all 200 Wikipedia items


### 6. BM25 + hybrid retrieval (unchanged from earlier notebooks)

In [6]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
_bm = {}
def bm25_scores(q):
    if q not in _bm:
        _bm[q] = np.asarray(bm25.get_scores(tokenize(q)))
    return _bm[q]

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

print("BM25 index built")

BM25 index built


### 7. Fixed-shape batching — the part that makes TPU viable

`batched_fixed` pads the final partial batch up to full size by repeating its last element, then
discards the padding from the results. Combined with `padding="max_length"`, every tensor entering
a model has shape `(batch_size, max_length)`, so XLA compiles one graph per model rather than one
per distinct shape.

In [7]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

def batched_fixed(items, bs):
    """Yield (padded_batch, n_real). Last batch is padded up to bs for XLA."""
    for i in range(0, len(items), bs):
        chunk = list(items[i:i + bs])
        n = len(chunk)
        if n < bs:
            chunk += [chunk[-1]] * (bs - n)   # shape-stable padding, trimmed after
        yield chunk, n


def mean_pool(last_hidden, mask):
    m = mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


@torch.no_grad()
def encode_texts(model, tok, texts, bs, max_len, label=""):
    """e5-style: mean pooling + L2 normalisation, on the selected device."""
    out, done, t0 = [], 0, time.time()
    for chunk, n in batched_fixed(texts, bs):
        enc = tok(chunk, padding="max_length", truncation=True,
                  max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        h = model(**enc).last_hidden_state
        v = mean_pool(h, enc["attention_mask"])
        v = torch.nn.functional.normalize(v, p=2, dim=1)
        sync()
        out.append(v.float().cpu().numpy()[:n])
        done += n
        if done % (bs * 10) < bs:
            print(f"    {label} {done}/{len(texts)}  ({time.time() - t0:.0f}s)", flush=True)
    return np.concatenate(out, 0).astype("float32")


@torch.no_grad()
def score_pairs(model, tok, pairs, bs, max_len, label=""):
    """Cross-encoder relevance score for each (query, passage) pair."""
    out, done, t0 = [], 0, time.time()
    for chunk, n in batched_fixed(pairs, bs):
        enc = tok([a for a, _ in chunk], [b for _, b in chunk],
                  padding="max_length", truncation=True,
                  max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        sync()
        s = logits.float().cpu().numpy()
        # Rerankers emit a single relevance logit; classifiers emit 2 (take "relevant").
        s = s[:, 0] if s.shape[-1] == 1 else s[:, -1]
        out.extend(s[:n].tolist())
        done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(pairs)}  ({time.time() - t0:.0f}s)", flush=True)
    return np.asarray(out)

print("helpers ready")

helpers ready


### 8. Stage 1 — retrieve candidates, then free the bi-encoder

In [8]:
print(f"Building index on {BACKEND.upper()}...")
t0 = time.time()

bi_tok = AutoTokenizer.from_pretrained(CONFIG["base_encoder"])
# Cast after loading rather than passing torch_dtype=: that kwarg is deprecated in
# newer transformers and absent in older ones, while .to(dtype=) works on every version.
bi = AutoModel.from_pretrained(CONFIG["base_encoder"]).to(device=device, dtype=torch.float32).eval()

corpus_emb = encode_texts(bi, bi_tok, [f"passage: {t}" for t in corpus_texts],
                          CONFIG["encode_batch_size"], CONFIG["max_length"], "corpus")
print(f"  corpus encoded in {time.time() - t0:.0f}s")

q_fields = ["msa_query", "darija_query"]
query_emb = {}
for field in q_fields:
    query_emb[field] = encode_texts(bi, bi_tok, [f"query: {q[field]}" for q in qa],
                                    CONFIG["encode_batch_size"], CONFIG["max_length"], field)
    print(f"  {field} encoded")

K = CONFIG["rerank_k"]
candidates = {}
for field in q_fields:
    d = {}
    for i, q in enumerate(qa):
        s = (CONFIG["alpha"] * minmax(corpus_emb @ query_emb[field][i])
             + (1 - CONFIG["alpha"]) * minmax(bm25_scores(q[field])))
        d[q["id"]] = [corpus_ids[j] for j in np.argsort(-s)[:K]]
    candidates[field] = d
    print(f"  {field} candidates done")

del bi, corpus_emb, query_emb
gc.collect()
if BACKEND == "cuda":
    torch.cuda.empty_cache()
print(f"Stage 1 total {time.time() - t0:.0f}s")

Building index on CPU...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    corpus 320/3054  (42s)
    corpus 640/3054  (84s)
    corpus 960/3054  (127s)
    corpus 1280/3054  (170s)
    corpus 1600/3054  (213s)
    corpus 1920/3054  (256s)
    corpus 2240/3054  (299s)
    corpus 2560/3054  (342s)
    corpus 2880/3054  (385s)
  corpus encoded in 424s
  msa_query encoded
  darija_query encoded
  msa_query candidates done
  darija_query candidates done
Stage 1 total 487s


### 9. The ceiling: what is the best a reranker could possibly do?

In [ ]:
print("=" * 78)
print(f"CEILING ANALYSIS - Recall@{K} of the retrieval stage")
print("=" * 78)
print("A reranker can only reorder what retrieval already returned. Recall@K is")
print("therefore a hard upper bound on post-rerank Recall@1.\n")

ceiling = {}
for field in q_fields:
    hits = [int(q["source_chunk_id"] in candidates[field][q["id"]]) for q in qa]
    ceiling[field] = float(np.mean(hits))
    print(f"  {field:<14} Recall@{K} = {ceiling[field]:.3f}   <- reranking ceiling")

CEILING ANALYSIS - Recall@20 of the retrieval stage
A reranker can only reorder what retrieval already returned. Recall@K is
therefore a hard upper bound on post-rerank Recall@1.

  msa_query      Recall@20 = 0.995   <- reranking ceiling
  darija_query   Recall@20 = 0.970   <- reranking ceiling


### 10. Baseline (no reranking) for comparison

In [ ]:
def evaluate_order(ordered_ids_by_qid):
    """Per-item hit vectors from an ordered candidate list, for bootstrapping."""
    out = {f"R@{k}": [] for k in CONFIG["k_values"]}
    rr = []
    for q in qa:
        ordered = ordered_ids_by_qid[q["id"]]
        gold = q["source_chunk_id"]
        pos = ordered.index(gold) + 1 if gold in ordered else None
        for k in CONFIG["k_values"]:
            out[f"R@{k}"].append(1.0 if (pos is not None and pos <= k) else 0.0)
        rr.append(1.0 / pos if pos else 0.0)
    return {**{k: np.array(v) for k, v in out.items()}, "MRR": np.array(rr)}

results = {}
for field in q_fields:
    results[("no_rerank", field)] = evaluate_order(candidates[field])

print("\nBaseline (retrieval order, no reranking):")
for field in q_fields:
    m = results[("no_rerank", field)]
    print(f"  {field:<14} R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}  MRR={m['MRR'].mean():.3f}")

print(f"\nDarija headroom available to reranking: "
      f"{ceiling['darija_query'] - results[('no_rerank','darija_query')]['R@1'].mean():+.3f}")


Baseline (retrieval order, no reranking):
  msa_query      R@1=0.755  R@5=0.950  MRR=0.837
  darija_query   R@1=0.620  R@5=0.905  MRR=0.742

Darija headroom available to reranking: +0.350


### 11. Resume from checkpoint — with a consistency guard

Checkpointed rows are per-item hit vectors scored against one specific candidate set. Reloading
rows produced by a *different* retrieval run and putting them in the same table would be quietly
wrong: the baseline they are compared against would not be the baseline they were measured with.

So the checkpoint's own `no_rerank` vectors are compared element-wise against the ones just
computed. If they differ, the reranker rows from that checkpoint are dropped and those models are
re-run against the current candidates.

In [ ]:
resumed, dropped = [], []

if CKPT_PATH:
    with open(CKPT_PATH, "rb") as f:
        ckpt = pickle.load(f)
    print(f"Loaded checkpoint: {CKPT_PATH}  ({len(ckpt)} entries)")

    # Does the checkpoint's baseline match the one this run just computed?
    consistent = True
    for field in q_fields:
        key = ("no_rerank", field)
        if key not in ckpt:
            consistent = False
            print(f"  checkpoint has no baseline for {field}")
            break
        for metric in ["R@1", "MRR"]:
            a, b = np.asarray(ckpt[key][metric]), np.asarray(results[key][metric])
            if a.shape != b.shape or not np.allclose(a, b):
                consistent = False
                print(f"  baseline MISMATCH on {field}/{metric}: "
                      f"checkpoint mean {a.mean():.3f} vs this run {b.mean():.3f}")
                break
        if not consistent:
            break

    if consistent:
        print("  baseline matches -- checkpointed reranker rows are comparable, reusing them.")
        for k, v in ckpt.items():
            if k[0] != "no_rerank":
                results[k] = v
                if k[1] == "darija_query":
                    resumed.append(k[0])
    else:
        print("\n  Checkpoint came from a DIFFERENT retrieval configuration.")
        if CONFIG["strict_resume"]:
            print("  strict_resume=True -> discarding its reranker rows and re-running those models")
            dropped = sorted({k[0] for k in ckpt if k[0] != "no_rerank"})
        else:
            print("  strict_resume=False -> reusing anyway (rows will NOT be comparable)")
            for k, v in ckpt.items():
                if k[0] != "no_rerank":
                    results[k] = v
                    if k[1] == "darija_query":
                        resumed.append(k[0])
else:
    print("No checkpoint found.")

# Decide the final run list.
run_list = list(CONFIG["rerankers"])
completed_short = CONFIG["completed_model"].split("/")[-1]
if CONFIG["auto_include_completed"] and completed_short not in resumed:
    run_list.insert(0, CONFIG["completed_model"])
    why = "checkpoint discarded as inconsistent" if dropped else "no checkpoint supplied it"
    print(f"\nAdding {completed_short} back to the run ({why}),")
    print("so the results table and headline are not missing the model that produced them.")

print("\n" + "=" * 78)
print("MODELS TO RUN:")
for n in run_list:
    print("   -", n)
print("REUSED FROM CHECKPOINT:", ", ".join(resumed) if resumed else "(none)")
print("=" * 78)

Loaded checkpoint: rerank_results_partial.pkl  (8 entries)
  baseline MISMATCH on msa_query/R@1: checkpoint mean 0.760 vs this run 0.755

  Checkpoint came from a DIFFERENT retrieval configuration.
  strict_resume=True -> discarding its reranker rows and re-running those models

Adding bge-reranker-v2-m3 back to the run (checkpoint discarded as inconsistent),
so the results table and headline are not missing the model that produced them.

MODELS TO RUN:
   - BAAI/bge-reranker-v2-m3
   - cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
   - BAAI/bge-reranker-base
REUSED FROM CHECKPOINT: (none)


### 12. Stage 2 — cross-encoder reranking on the TPU

In [12]:
RESULTS_CACHE = "rerank_results_partial.pkl"

def save_results():
    # Checkpoint after every model, not just at the end -- a failure in model N
    # must not cost the results already obtained from models 1..N-1.
    with open(RESULTS_CACHE, "wb") as f:
        pickle.dump(results, f)
    if DRIVE_DIR:
        with open(os.path.join(DRIVE_DIR, RESULTS_CACHE), "wb") as f:
            pickle.dump(results, f)

def rerank_with(model_name):
    """Score every (query, candidate) pair jointly and reorder.

    The ENTIRE body is guarded, not just model loading: a device-specific failure
    inside the forward pass must not kill the run and lose earlier models' results.
    """
    try:
        tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        ce = AutoModelForSequenceClassification.from_pretrained(
            model_name, trust_remote_code=True).to(device=device, dtype=torch.float32).eval()
    except Exception as e:
        print(f"  SKIPPED (load failed) {model_name}: {type(e).__name__}: {str(e)[:200]}")
        return None

    out = {}
    try:
        for field in q_fields:
            qids = [q["id"] for q in qa]
            # Flatten to one long uniform stream; first batch pays the XLA compile.
            flat = [(q[field], corpus_map[c])
                    for q in qa for c in candidates[field][q["id"]]]
            t0 = time.time()
            scores = score_pairs(ce, tok, flat, CONFIG["batch_size"],
                                 CONFIG["max_length"], f"{field}")
            reordered = {}
            for i, qid in enumerate(qids):
                cands = candidates[field][qid]
                s = scores[i * K:(i + 1) * K]
                reordered[qid] = [cands[j] for j in np.argsort(-s)]
            out[field] = reordered
            print(f"    {field} reranked ({len(flat)} pairs, {time.time() - t0:.0f}s)")
    except Exception as e:
        print(f"  SKIPPED (inference failed) {model_name}: {type(e).__name__}: {str(e)[:200]}")
        del ce
        gc.collect()
        return None

    del ce
    gc.collect()
    if BACKEND == "cuda":
        torch.cuda.empty_cache()
    return out


for name in run_list:
    short = name.split("/")[-1]
    if (short, "darija_query") in results:
        print(f"\n=== {short} === (already have results, skipping)")
        continue
    print(f"\n=== {short} ===  on {BACKEND.upper()}")
    reordered = rerank_with(name)
    if reordered is None:
        continue
    for field in q_fields:
        results[(short, field)] = evaluate_order(reordered[field])
    save_results()
    m = results[(short, "darija_query")]
    print(f"  Darija R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}  MRR={m['MRR'].mean():.3f}")


=== bge-reranker-v2-m3 ===  on CPU


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

    msa_query 640/4000  (273s)
    msa_query 1280/4000  (548s)
    msa_query 1920/4000  (824s)
    msa_query 2560/4000  (1101s)
    msa_query 3200/4000  (1377s)
    msa_query 3840/4000  (1654s)
    msa_query reranked (4000 pairs, 1724s)
    darija_query 640/4000  (280s)
    darija_query 1280/4000  (559s)
    darija_query 1920/4000  (838s)
    darija_query 2560/4000  (1118s)
    darija_query 3200/4000  (1397s)
    darija_query 3840/4000  (1676s)
    darija_query reranked (4000 pairs, 1746s)
  Darija R@1=0.800  R@5=0.960  MRR=0.872

=== mmarco-mMiniLMv2-L12-H384-v1 ===  on CPU


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

    msa_query 640/4000  (29s)
    msa_query 1280/4000  (59s)
    msa_query 1920/4000  (88s)
    msa_query 2560/4000  (117s)
    msa_query 3200/4000  (146s)
    msa_query 3840/4000  (176s)
    msa_query reranked (4000 pairs, 183s)
    darija_query 640/4000  (29s)
    darija_query 1280/4000  (59s)
    darija_query 1920/4000  (88s)
    darija_query 2560/4000  (117s)
    darija_query 3200/4000  (147s)
    darija_query 3840/4000  (176s)
    darija_query reranked (4000 pairs, 184s)
  Darija R@1=0.715  R@5=0.925  MRR=0.810

=== bge-reranker-base ===  on CPU


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

    msa_query 640/4000  (87s)
    msa_query 1280/4000  (175s)
    msa_query 1920/4000  (260s)
    msa_query 2560/4000  (348s)
    msa_query 3200/4000  (435s)
    msa_query 3840/4000  (522s)
    msa_query reranked (4000 pairs, 543s)
    darija_query 640/4000  (86s)
    darija_query 1280/4000  (173s)
    darija_query 1920/4000  (259s)
    darija_query 2560/4000  (346s)
    darija_query 3200/4000  (433s)
    darija_query 3840/4000  (520s)
    darija_query reranked (4000 pairs, 541s)
  Darija R@1=0.365  R@5=0.720  MRR=0.527


### 13. Results table

In [13]:
rows = []
for (method, field), m in results.items():
    rows.append({"method": method, "query": field,
                 **{k: float(v.mean()) for k, v in m.items()}})
df = pd.DataFrame(rows)
df.to_csv("rerank_results.csv", index=False)

print("\n" + "=" * 78)
print(f"RESULTS   (backend: {BACKEND.upper()})")
print("=" * 78)
pivot = df.pivot(index="method", columns="query", values=["R@1", "R@5", "MRR"])
print(pivot.to_string(float_format=lambda x: f"{x:.3f}"))


RESULTS   (backend: CPU)
                                      R@1                    R@5                    MRR          
query                        darija_query msa_query darija_query msa_query darija_query msa_query
method                                                                                           
bge-reranker-base                   0.365     0.520        0.720     0.845        0.527     0.661
bge-reranker-v2-m3                  0.800     0.875        0.960     0.995        0.872     0.925
mmarco-mMiniLMv2-L12-H384-v1        0.715     0.865        0.925     0.970        0.810     0.913
no_rerank                           0.620     0.755        0.905     0.950        0.742     0.837


### 14. Improvement over no reranking, with paired bootstrap CIs

In [14]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a, b):
    d = np.asarray(a) - np.asarray(b)
    idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
    boot = d[idx].mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return d.mean(), lo, hi

print("\n" + "=" * 78)
print("RERANKING GAIN over retrieval order (paired 95% CI)")
print("=" * 78)
gain_rows = []
for method in df.method.unique():
    if method == "no_rerank":
        continue
    for field in q_fields:
        if (method, field) not in results:
            continue
        for metric in ["R@1", "MRR"]:
            d, lo, hi = paired(results[(method, field)][metric],
                               results[("no_rerank", field)][metric])
            sig = "yes" if lo > 0 else ("worse" if hi < 0 else "no")
            gain_rows.append({"method": method, "query": field, "metric": metric,
                              "gain": d, "lo": lo, "hi": hi, "significant": sig})
gains = pd.DataFrame(gain_rows)
print(gains.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gains.to_csv("rerank_gains.csv", index=False)


RERANKING GAIN over retrieval order (paired 95% CI)
                      method        query metric   gain     lo     hi significant
          bge-reranker-v2-m3    msa_query    R@1 +0.120 +0.075 +0.170         yes
          bge-reranker-v2-m3    msa_query    MRR +0.088 +0.058 +0.120         yes
          bge-reranker-v2-m3 darija_query    R@1 +0.180 +0.120 +0.245         yes
          bge-reranker-v2-m3 darija_query    MRR +0.131 +0.094 +0.175         yes
mmarco-mMiniLMv2-L12-H384-v1    msa_query    R@1 +0.110 +0.060 +0.160         yes
mmarco-mMiniLMv2-L12-H384-v1    msa_query    MRR +0.076 +0.043 +0.110         yes
mmarco-mMiniLMv2-L12-H384-v1 darija_query    R@1 +0.095 +0.030 +0.165         yes
mmarco-mMiniLMv2-L12-H384-v1 darija_query    MRR +0.069 +0.027 +0.109         yes
           bge-reranker-base    msa_query    R@1 -0.235 -0.315 -0.155       worse
           bge-reranker-base    msa_query    MRR -0.177 -0.227 -0.120       worse
           bge-reranker-base darija_query    

### 15. Effect on the dialect gap, which is what the project is about

In [15]:
print("\n" + "=" * 78)
print("EFFECT ON THE DIALECT GAP (MSA - Darija)")
print("=" * 78)
gap_rows = []
for method in df.method.unique():
    if (method, "msa_query") not in results:
        continue
    for metric in ["R@1", "MRR"]:
        d, lo, hi = paired(results[(method, "msa_query")][metric],
                           results[(method, "darija_query")][metric])
        gap_rows.append({"method": method, "metric": metric, "gap": d,
                         "lo": lo, "hi": hi,
                         "gap_significant": "yes" if lo > 0 else "no"})
gapdf = pd.DataFrame(gap_rows).sort_values(["metric", "gap"])
print(gapdf.to_string(index=False, float_format=lambda x: f"{x:+.3f}"))
gapdf.to_csv("rerank_dialect_gap.csv", index=False)

print("""
Two outcomes are both worth reporting:
  - Gap SHRINKS  -> reranking mitigates dialect mismatch; a practical recommendation
                    that needs no training data at all.
  - Gap PERSISTS -> the penalty is not merely a ranking artefact of the bi-encoder;
                    it survives a much stronger ranking model, which is a stronger
                    claim about the phenomenon than anything measured so far.
""")


EFFECT ON THE DIALECT GAP (MSA - Darija)
                      method metric    gap     lo     hi gap_significant
          bge-reranker-v2-m3    MRR +0.053 +0.026 +0.081             yes
                   no_rerank    MRR +0.096 +0.065 +0.127             yes
mmarco-mMiniLMv2-L12-H384-v1    MRR +0.103 +0.069 +0.142             yes
           bge-reranker-base    MRR +0.133 +0.085 +0.179             yes
          bge-reranker-v2-m3    R@1 +0.075 +0.040 +0.120             yes
                   no_rerank    R@1 +0.135 +0.090 +0.180             yes
mmarco-mMiniLMv2-L12-H384-v1    R@1 +0.150 +0.095 +0.210             yes
           bge-reranker-base    R@1 +0.155 +0.080 +0.230             yes

Two outcomes are both worth reporting:
  - Gap SHRINKS  -> reranking mitigates dialect mismatch; a practical recommendation
                    that needs no training data at all.
  - Gap PERSISTS -> the penalty is not merely a ranking artefact of the bi-encoder;
                    it survives a mu

### 16. Headline

In [16]:
best = df[(df["query"] == "darija_query") & (df.method != "no_rerank")]
if len(best):
    top = best.loc[best["R@1"].idxmax()]
    base_r1 = df[(df.method == "no_rerank") & (df["query"] == "darija_query")]["R@1"].iloc[0]
    print("=" * 78)
    print(f"BEST RERANKER: {top['method']}")
    print("=" * 78)
    print(f"  Darija R@1   {base_r1:.3f} -> {top['R@1']:.3f}   ({top['R@1'] - base_r1:+.3f})")
    print(f"  Ceiling (Recall@{K})            {ceiling['darija_query']:.3f}")
    print(f"  Headroom captured              "
          f"{(top['R@1'] - base_r1) / max(ceiling['darija_query'] - base_r1, 1e-9) * 100:.1f}%")
    print(f"  Hardware                       {BACKEND.upper()}")
    print(f"  Models this run                {', '.join(m.split('/')[-1] for m in run_list)}")
    print(f"  Reused from checkpoint         {', '.join(resumed) if resumed else '(none)'}")
else:
    print("No reranker loaded successfully - check the model names in CONFIG.")

BEST RERANKER: bge-reranker-v2-m3
  Darija R@1   0.620 -> 0.800   (+0.180)
  Ceiling (Recall@20)            0.970
  Headroom captured              51.4%
  Hardware                       CPU
  Models this run                bge-reranker-v2-m3, mmarco-mMiniLMv2-L12-H384-v1, bge-reranker-base
  Reused from checkpoint         (none)


### 17. Persist the CSVs

Copies the result CSVs into Drive alongside the checkpoint. The executed notebook itself is saved
by Colab's autosave, so `File → Save` writes this file back to Drive **with all outputs included**.

In [17]:
import shutil, glob

if DRIVE_DIR:
    for f in glob.glob("rerank_*.csv") + glob.glob("rerank_results_partial.pkl"):
        shutil.copy(f, DRIVE_DIR)
        print("saved", os.path.join(DRIVE_DIR, os.path.basename(f)))
else:
    print("Drive not mounted; CSVs are in the working dir only:", glob.glob("rerank_*.csv"))

print(f"\nRun complete on {BACKEND.upper()}.")
print("Now use File -> Save, then tell Claude the run is done.")

saved /content/drive/MyDrive/MSA_rerank_v3_tpu/rerank_gains.csv
saved /content/drive/MyDrive/MSA_rerank_v3_tpu/rerank_dialect_gap.csv
saved /content/drive/MyDrive/MSA_rerank_v3_tpu/rerank_results.csv
saved /content/drive/MyDrive/MSA_rerank_v3_tpu/rerank_results_partial.pkl

Run complete on CPU.
Now use File -> Save, then tell Claude the run is done.
